# MonteCarlo Analysis (Portfolio-safe)

This notebook demonstrates a portfolio-safe Monte Carlo workflow using synthetic/example inputs.
It shows: running simulations, final equity distribution, P10/P50/P90 bands, probability of ruin, and drawdown distributions.

**All data here is synthetic and for demonstration only. Do NOT include real strategy parameters.**


In [1]:
# Standard imports
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.stats import skewnorm, beta as beta_dist, norm

# For reproducibility
np.random.seed(42)
random.seed(42)


In [2]:
# Portfolio-safe Monte Carlo engine (simplified)

def rr_ratio_generator(N, mean, clip_min, clip_max, std=0.5, skew=0):
    # Returns a list of N rr samples (rounded) using a skewed normal truncated to [clip_min, clip_max]
    data = []
    while len(data) < N:
        new_data = skewnorm.rvs(a=skew, loc=mean, scale=std, size=N)
        new_data = new_data[(new_data >= clip_min) & (new_data <= clip_max)]
        data.extend(new_data)
    data = data[:N]
    return [round(float(x), 4) for x in data]


In [3]:
# Example synthetic inputs (safe defaults)
trials = 500
days = 252
initial_balance = 100000
max_loss = 2000
win_rate_obs = 0.55  # example observed win-rate (synthetic)

loss_rr = rr_ratio_generator(N=days, mean=1.03, clip_min=1.01, clip_max=1.15, skew=-1)
profit_rr = rr_ratio_generator(N=days, mean=1.95, clip_min=1.8, clip_max=2.1, skew=1)

print('sample loss_rr (first 10):', loss_rr[:10])
print('sample profit_rr (first 10):', profit_rr[:10])


sample loss_rr (first 10): [1.0325, 1.0383, 1.0555, 1.1172, 1.0736, 1.0193, 1.036, 1.1334, 1.0823, 1.0102]
sample profit_rr (first 10): [1.8592, 1.9798, 1.9325, 2.0953, 1.8926, 1.8783, 1.9835, 1.9533, 1.838, 2.0626]


In [4]:
# Monte Carlo with Beta + Bootstrap (safe, compact)
from scipy.stats import beta as beta_dist

def monte_carlo_with_beta_bootstrap(trials, days, initial_balance, win_rate_obs, loss_rr, profit_rr, max_loss, N_eff=200, miss_prob=0.03, shrinkage=True, shrinkage_strength=0.15):
    profit_rr_arr = np.array(profit_rr)
    loss_rr_arr = np.array(loss_rr)

    wins = int(round(win_rate_obs * N_eff))
    losses = N_eff - wins
    alpha = 1 + wins
    beta_param = 1 + losses
    sampled_win_rates = beta_dist.rvs(alpha, beta_param, size=trials)

    all_balances = []
    infos = []
    for i in range(trials):
        win_rate_i = float(sampled_win_rates[i])
        if shrinkage:
            win_rate_i = 0.5 + (1 - shrinkage_strength) * (win_rate_i - 0.5)

        balance = initial_balance
        daily_balances = [balance]
        daily_pnl = []
        for day in range(days):
            if random.random() > miss_prob:
                if np.random.rand() < win_rate_i:
                    rr = float(np.random.choice(profit_rr_arr))
                    pnl = rr * max_loss
                    balance += pnl
                    daily_pnl.append(pnl)
                else:
                    rr = float(np.random.choice(loss_rr_arr))
                    pnl = -rr * max_loss
                    balance += pnl
                    daily_pnl.append(pnl)
                daily_balances.append(balance)
            else:
                daily_pnl.append(0)
                daily_balances.append(balance)
        all_balances.append(daily_balances)
        infos.append(pd.DataFrame({'pnl': daily_pnl, 'balance': daily_balances[1:]}))

    balances_arr = np.array([np.asarray(b) for b in all_balances])
    return balances_arr, infos

# run
balances_arr, infos = monte_carlo_with_beta_bootstrap(trials, days, initial_balance, win_rate_obs, loss_rr, profit_rr, max_loss)
print('balances array shape:', balances_arr.shape)


balances array shape: (500, 253)


In [5]:
# Post-processing: final equity distribution, drawdowns, percentiles
final_equities = balances_arr[:, -1]

# drawdowns per trial
def compute_drawdowns(array_1d):
    peak = np.maximum.accumulate(array_1d)
    dd = (peak - array_1d) / peak
    dd[np.isnan(dd)] = 0.0
    return dd

max_drawdown_pcts = [compute_drawdowns(b).max() for b in balances_arr]

p10 = np.percentile(balances_arr, 10, axis=0)
p50 = np.percentile(balances_arr, 50, axis=0)
p90 = np.percentile(balances_arr, 90, axis=0)

print('Final equity percentiles (10/50/90):', np.percentile(final_equities, [10,50,90]))
print('Probability of ruin (<=0):', np.mean(final_equities <= 0))
print('Median max drawdown pct:', np.median(max_drawdown_pcts))


Final equity percentiles (10/50/90): [297554.58 379791.1  455186.32]
Probability of ruin (<=0): 0.0
Median max drawdown pct: 0.08617428920301508


In [6]:
# Plot percentile band + some sample paths (plotly)
import plotly.graph_objs as go
x = list(range(days+1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=list(p90), mode='lines', line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=x, y=list(p10), mode='lines', line=dict(width=0), fill='tonexty', fillcolor='rgba(200,200,200,0.3)', name='P10-P90'))
fig.add_trace(go.Scatter(x=x, y=list(p50), mode='lines', line=dict(color='orange', width=2), name='P50'))
# add 10 sample trials
for i in range(10):
    fig.add_trace(go.Scatter(x=x, y=list(balances_arr[i]), mode='lines', line=dict(width=1), opacity=0.6, showlegend=False))
fig.update_layout(title='P10-P90 bands + sample paths', xaxis_title='Day', yaxis_title='Balance')
fig.show()


### Interpretation
- P10–P90 band: realistic 80% central band of outcomes.
- P50: typical outcome (median).
- Probability of ruin: fraction of trials finishing at or below 0.

Use this notebook to demonstrate methodology with synthetic inputs. For public repo, keep these values synthetic.
